## 面试问题

探索 vs 利用：给循环一个探索预算，避免过早收敛或无限试探？

## 回答主线

纯利用(贪心)选第一个可行方案会过早收敛错过更优；纯探索无限试探不收敛。做法是给探索预算：预算内探索找更优、耗尽转利用。本 Notebook 用多个候选方案(质量未知需评估)，对比纯贪心(选第一个可行 planA 质量6)与探索预算=3(评估到 planC 质量9)。

## 真实案例

四个候选方案，第一个可行的 planA 质量 6，更优的 planC 质量 9 排在后面。纯贪心选 planA 就停；探索预算=3 的在预算内评估到 planC。数据为教学方案，不代表真实优化。

In [1]:
candidate_plans = [  # 候选方案列表，质量需逐个评估。
    {"name": "planA", "feasible": True, "quality": 6},  # 第一个可行方案质量中等。
    {"name": "planB", "feasible": False, "quality": 0},  # 不可行方案。
    {"name": "planC", "feasible": True, "quality": 9},  # 更优的可行方案。
    {"name": "planD", "feasible": True, "quality": 7},  # 另一个可行方案。
]  # 结束候选定义。

print("候选方案数:", len(candidate_plans))  # 展示候选数量。
for p in candidate_plans:  # 逐个打印候选。
    print("  ", p["name"], "feasible", p["feasible"], "quality", p["quality"])  # 展示每个方案。

候选方案数: 4
   planA feasible True quality 6
   planB feasible False quality 0
   planC feasible True quality 9
   planD feasible True quality 7


## 基线（Baseline）

反面基线：纯贪心（纯利用），选第一个可行方案就停。它抓住 planA(质量6)立即返回，错过后面质量 9 的 planC。

In [2]:
def run_greedy(plans):  # 纯利用：选第一个可行方案就停。
    evaluated = 0  # 记录评估次数。
    for p in plans:  # 逐个评估候选。
        evaluated += 1  # 每评估一个计数。
        if p["feasible"]:  # 第一个可行方案。
            return {"chosen": p["name"], "quality": p["quality"], "evaluated": evaluated}  # 立即选定返回。
    return {"chosen": None, "quality": 0, "evaluated": evaluated}  # 无可行方案。

greedy = run_greedy(candidate_plans)  # 运行纯贪心。
print("纯贪心结果:", greedy)  # 展示选了第一个可行的 planA 质量6 错过更优。

纯贪心结果: {'chosen': 'planA', 'quality': 6, 'evaluated': 1}


## 失败案例与修正

贪心过早收敛错过更优。修正是探索预算：预算内评估多个候选、保留当前最优，预算耗尽转利用。探索预算=3 时评估到 planC(质量9)。

In [3]:
def run_with_exploration(plans, explore_budget=3):  # 有探索预算：预算内多评估几个再选最优。
    evaluated = 0  # 记录评估次数。
    best = {"chosen": None, "quality": -1}  # 记录当前最优。
    for p in plans:  # 逐个评估候选。
        if evaluated >= explore_budget:  # 探索预算耗尽转为利用。
            break  # 停止探索。
        evaluated += 1  # 每评估一个计数。
        if p["feasible"] and p["quality"] > best["quality"]:  # 发现更优的可行方案。
            best = {"chosen": p["name"], "quality": p["quality"]}  # 更新最优。
    best["evaluated"] = evaluated  # 记录总评估次数。
    return best  # 返回预算内的最优方案。

In [4]:
explored = run_with_exploration(candidate_plans, explore_budget=3)  # 用探索预算=3 运行。
print("探索预算结果:", explored)  # 展示在预算内评估到 planC 质量9。
print("纯贪心 选", greedy["chosen"], "质量", greedy["quality"], "评估", greedy["evaluated"], "次")  # 贪心质量低。
print("探索预算 选", explored["chosen"], "质量", explored["quality"], "评估", explored["evaluated"], "次")  # 探索质量高。

探索预算结果: {'chosen': 'planC', 'quality': 9, 'evaluated': 3}
纯贪心 选 planA 质量 6 评估 1 次
探索预算 选 planC 质量 9 评估 3 次


In [5]:
print("质量提升:", explored["quality"] - greedy["quality"])  # 展示探索带来的质量提升。
print("多评估次数:", explored["evaluated"] - greedy["evaluated"])  # 展示探索的额外成本。
print("探索是否在预算内:", explored["evaluated"] <= 3)  # 展示探索不超预算。

质量提升: 3
多评估次数: 2
探索是否在预算内: True


## 结果解读

纯贪心评估 1 次选 planA(质量6)；探索预算=3 评估 3 次找到 planC(质量9)，以 2 次额外评估换来质量提升 3。要点：探索有预算且衰减、贪心的代价是错过更优、探索也消耗资源、预算大小用回归集调。

In [6]:
assert greedy["chosen"] == "planA"  # 纯贪心选第一个可行方案。
assert greedy["quality"] == 6  # 纯贪心质量为 6。
assert explored["chosen"] == "planC"  # 探索预算找到更优方案。
assert explored["quality"] == 9  # 探索预算质量为 9。
assert explored["evaluated"] > greedy["evaluated"]  # 探索以更多评估换取更高质量。
assert explored["evaluated"] <= 3  # 探索不超过预算。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
